# Silver Layer - Data Transformation

The Silver layer cleans and standardizes Bronze data while preserving each source table's useful grain. It does not create business KPIs or wide joined tables.

## Transformation plan

| Table | Source grain | Silver transformation | Gold usage |
|---|---|---|---|
| customers | One customer record per `customer_id` | Trim identifiers, cast ZIP prefix, remove exact duplicates, retain both customer identifiers | Customer geography and customer-level metrics |
| orders | One row per `order_id` | Trim identifiers/status, cast all timestamps, remove exact duplicates | Order-level sales and delivery analytics |
| order_items | One item row within an order | Cast sequence and numeric columns, remove exact duplicates only | Order, product, and seller aggregations |
| order_payments | One payment record within an order | Cast numeric fields, preserve multiple payment rows, remove exact duplicates only | Aggregate to order before sales joins |
| order_reviews | Review record related to an order | Cast score, preserve missing comments and missing reviews, remove exact duplicates only | Aggregate review data before order joins |
| products | One row per `product_id` | Cast dimensions/weight, preserve nullable product attributes | Product and category analytics |
| sellers | One row per `seller_id` | Trim identifiers, cast ZIP prefix, remove exact duplicates | Seller analytics and seller geography |
| geolocation | Many observations per ZIP prefix | Cast coordinates and aggregate to ZIP-prefix representatives; repeated ZIP rows are legitimate | Optional geographic enrichment |

Important: keys are not deduplicated blindly. A repeated `order_id` in payments or a repeated ZIP prefix in geolocation can represent valid source records.

## Objective

In [ ]:
from functools import reduce
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

spark = (
    SparkSession.builder
    .appName("Olist-Silver-Transformation")
    .getOrCreate()
)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

BRONZE_PATH = "../../data/bronze"
SILVER_PATH = "../../data/silver"
TABLES = ["customers", "orders", "order_items", "order_payments", "order_reviews", "products", "sellers", "geolocation"]
print(f"Spark: {spark.version}")

## Load Bronze Data

The pipeline reads only Parquet from Bronze. If Bronze is incomplete, stop here and rerun the Bronze ingestion notebook first.

In [ ]:
def bronze_table_exists(table_name):
    return os.path.exists(os.path.join(BRONZE_PATH, table_name))

missing_tables = [table for table in TABLES if not bronze_table_exists(table)]
if missing_tables:
    raise FileNotFoundError(
        f"Missing Bronze tables: {missing_tables}. Run 01_bronze_ingestion.ipynb first."
    )

bronze = {table: spark.read.parquet(os.path.join(BRONZE_PATH, table)) for table in TABLES}
bronze_counts = {table: frame.count() for table, frame in bronze.items()}
bronze_counts

## Inspect Bronze Schemas

In [ ]:
for table, frame in bronze.items():
    print(f"{table}: {frame.count():,} rows, {len(frame.columns)} columns")
    frame.printSchema()

In [ ]:
def trim_string_columns(frame):
    string_columns = [field.name for field in frame.schema.fields if field.dataType.simpleString() == "string"]
    return reduce(
        lambda result, column_name: result.withColumn(column_name, F.trim(F.col(column_name))),
        string_columns,
    ) if False else reduce(
        lambda result, column_name: result.withColumn(column_name, F.trim(F.col(column_name))),
        string_columns,
        frame,
    )

def clean_exact_duplicates(frame):
    return frame.dropDuplicates()

def cast_if_present(frame, column_name, data_type):
    return frame.withColumn(column_name, F.col(column_name).cast(data_type)) if column_name in frame.columns else frame

def standardize(frame, casts=None):
    result = trim_string_columns(frame)
    for column_name, data_type in (casts or {}).items():
        result = cast_if_present(result, column_name, data_type)
    return clean_exact_duplicates(result)

## Customer Transformation

`customer_unique_id` is the business-level customer identifier. `customer_id` remains available as the order relationship key.

In [ ]:
customers = standardize(
    bronze["customers"],
    {"customer_zip_code_prefix": "int"},
)
customers = customers.filter(F.col("customer_id").isNotNull())
customers = customers.select(
    "customer_id", "customer_unique_id", "customer_zip_code_prefix",
    "customer_city", "customer_state",
    *[column for column in ["_ingested_at", "_source"] if column in customers.columns],
)
customers.show(3, truncate=False)

## Order Transformation

In [ ]:
order_timestamps = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders = standardize(
    bronze["orders"],
    {column: "timestamp" for column in order_timestamps},
)
orders = orders.filter(F.col("order_id").isNotNull())
orders = orders.filter(F.col("customer_id").isNotNull())

## Order Item Transformation

In [ ]:
order_items = standardize(
    bronze["order_items"],
    {"order_item_id": "int", "shipping_limit_date": "timestamp", "price": "double", "freight_value": "double"},
)
order_items = order_items.filter(
    F.col("order_id").isNotNull() & F.col("product_id").isNotNull() & F.col("seller_id").isNotNull()
)
order_items = order_items.filter((F.col("price") >= 0) & (F.col("freight_value") >= 0))

## Payment Transformation

Payment rows are intentionally not reduced to one row per order here. Multiple payment records can be legitimate.

In [ ]:
order_payments = standardize(
    bronze["order_payments"],
    {"payment_sequential": "int", "payment_installments": "int", "payment_value": "double"},
)
order_payments = order_payments.filter(F.col("order_id").isNotNull())
order_payments = order_payments.filter((F.col("payment_value") >= 0) & (F.col("payment_installments") > 0))

## Review Transformation

A missing review is not a zero review. Missing review fields remain null.

In [ ]:
order_reviews = standardize(
    bronze["order_reviews"],
    {"review_score": "int", "review_creation_date": "timestamp", "review_answer_timestamp": "timestamp"},
)
order_reviews = order_reviews.filter(F.col("order_id").isNotNull())
order_reviews = order_reviews.filter(F.col("review_score").isNull() | F.col("review_score").between(1, 5))

## Product Transformation

In [ ]:
product_numeric = {
    "product_name_lenght": "int", "product_description_lenght": "int",
    "product_photos_qty": "int", "product_weight_g": "double",
    "product_length_cm": "double", "product_height_cm": "double",
    "product_width_cm": "double",
}
products = standardize(bronze["products"], product_numeric)
products = products.filter(F.col("product_id").isNotNull())

## Seller Transformation

In [ ]:
sellers = standardize(bronze["sellers"], {"seller_zip_code_prefix": "int"})
sellers = sellers.filter(F.col("seller_id").isNotNull())

## Geolocation Transformation

There are many records per ZIP prefix. Those records are not exact duplicates, so they are summarized to a representative ZIP-prefix location. The row count is expected to decrease and `source_row_count` documents that change.

In [ ]:
geolocation_raw = standardize(
    bronze["geolocation"],
    {"geolocation_zip_code_prefix": "int", "geolocation_lat": "double", "geolocation_lng": "double"},
)
geolocation_raw = geolocation_raw.filter(
    F.col("geolocation_zip_code_prefix").isNotNull()
    & F.col("geolocation_lat").between(-35, 6)
    & F.col("geolocation_lng").between(-75, -30)
)
geolocation = (
    geolocation_raw.groupBy("geolocation_zip_code_prefix")
    .agg(
        F.avg("geolocation_lat").alias("representative_lat"),
        F.avg("geolocation_lng").alias("representative_lng"),
        F.count("*").alias("source_row_count"),
    )
)

## Silver Validation

In [ ]:
silver = {
    "customers": customers, "orders": orders, "order_items": order_items,
    "order_payments": order_payments, "order_reviews": order_reviews,
    "products": products, "sellers": sellers, "geolocation": geolocation,
}
validation_rows = []
for table, frame in silver.items():
    key_columns = {
        "customers": ["customer_id"], "orders": ["order_id"],
        "order_items": ["order_id", "order_item_id"],
        "order_payments": ["order_id", "payment_sequential"],
        "order_reviews": ["review_id"], "products": ["product_id"],
        "sellers": ["seller_id"], "geolocation": ["geolocation_zip_code_prefix"],
    }[table]
    duplicate_keys = frame.groupBy(*key_columns).count().filter(F.col("count") > 1).count()
    critical_nulls = sum(frame.filter(F.col(column).isNull()).count() for column in key_columns if column in frame.columns)
    validation_rows.append((table, bronze_counts[table], frame.count(), len(frame.columns), duplicate_keys, critical_nulls))

validation = spark.createDataFrame(
    validation_rows,
    ["table", "bronze_rows", "silver_rows", "silver_columns", "duplicate_key_groups", "critical_null_values"],
)
validation.orderBy("table").show(truncate=False)

## Write Silver Layer

In [ ]:
for table, frame in silver.items():
    (
        frame.write
        .mode("overwrite")
        .parquet(os.path.join(SILVER_PATH, table))
    )
print(f"Wrote {len(silver)} Silver datasets to {SILVER_PATH}")

## Summary

Silver preserves source grains, removes only exact duplicate rows, validates critical keys, and keeps semantically meaningful nulls. Geolocation is the deliberate exception: it is represented once per ZIP prefix using average coordinates and a source-row count.